# Semana 1: carga y análisis del rendimiento estudiantil

En este notebook se desarrolla un flujo de trabajo con datos académicos obtenidos desde Hugging Face mediante el dataset `mstz/student_performance`.

El proceso incluye:

1. Importar las clases reutilizables del proyecto.
2. Descargar el dataset y convertirlo en un DataFrame de pandas.
3. Examinar la estructura, los tipos de datos y los valores faltantes.
4. Generar un reporte en pantalla y guardarlo en un archivo `.txt`.
5. Aplicar cuatro transformaciones al DataFrame para preparar y analizar los datos de rendimiento estudiantil.

## 1. Preparar el entorno e importar la clase de carga

El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar `src/clases_po` y se agrega `src` a `sys.path`. Así se puede importar `HFDatasetLoader` sin instalar el proyecto como paquete.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

/home/runner/work/AD_ATV-1/AD_ATV-1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configurar el dataset y la salida

`dataset_name` identifica el repositorio en Hugging Face y `split="train"` indica la división que se descargará. El parámetro `filename` hace que el `DataFrame` también se guarde como CSV en `src/sincronicas/datos/` cuando el notebook se ejecuta desde esa carpeta.

In [2]:
loader = HFDatasetLoader(
    dataset_name="mstz/student_performance",
    split="train",
    filename="datos/student_performance.csv",
)

## 3. Descargar el dataset y crear el DataFrame

El método `load()` descarga el dataset, lo convierte a `pandas.DataFrame` y lo asigna a la variable `df`. La expresión `df.head()` muestra las primeras filas para comprobar que la carga fue exitosa.

In [3]:
df = loader.load()
df.head()

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 1000/1000 [00:00<00:00, 165117.08 examples/s]

,is_male,ethnicity,parental_level_of_education,has_standard_lunch,has_completed_preparation_test,reading_score,writing_score,has_passed_math_exam
0,False,group D,2,True,True,70,78,0
1,True,group D,5,True,False,93,87,1
2,False,group D,2,False,False,76,77,0
3,True,group B,2,False,False,70,63,1
4,False,group D,5,True,False,85,86,1


## 4. Revisar la estructura del DataFrame

Esta celda realiza una revisión inicial: cuenta filas y columnas y muestra el tipo de cada columna junto con información de memoria y valores no nulos. Esta inspección ayuda a detectar problemas antes de analizar los datos.

In [4]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
df.info()

Filas: 1000
Columnas: 8
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   is_male                         1000 non-null   bool  
 1   ethnicity                       1000 non-null   object
 2   parental_level_of_education     1000 non-null   int64 
 3   has_standard_lunch              1000 non-null   bool  
 4   has_completed_preparation_test  1000 non-null   bool  
 5   reading_score                   1000 non-null   int64 
 6   writing_score                   1000 non-null   int64 
 7   has_passed_math_exam            1000 non-null   int64 
dtypes: bool(3), int64(4), object(1)
memory usage: 42.1+ KB


## 5. Importar la clase de reportes

`Reportes` recibe un `DataFrame` en sus métodos y permite generar un resumen textual, imprimirlo en la consola o guardarlo en un archivo `.txt`.

In [5]:
from clases_po import Reportes

## 5.1. Transformar el DataFrame

Una vez importadas las clases necesarias, se aplica una transformación mínima al conjunto de datos. Se filtrarán los registros de estudiantes cuya puntuación en `reading_score` sea mayor o igual a 80.

El resultado se almacenará en `df_transformado`, conservando intacto el DataFrame original. Este nuevo DataFrame será utilizado en las siguientes secciones para generar, visualizar y guardar el reporte descriptivo.

In [6]:
# Filtrar los estudiantes cuya puntuación de lectura sea mayor o igual a 80.
df_transformado = df[df["reading_score"] >= 80].copy()

# Comparar la cantidad de registros antes y después de aplicar el filtro.
print(f"Filas originales: {len(df)}")
print(f"Filas después de la transformación: {len(df_transformado)}")

# Mostrar las primeras filas del DataFrame transformado.
df_transformado.head()

Filas originales: 1000
Filas después de la transformación: 261


,is_male,ethnicity,parental_level_of_education,has_standard_lunch,has_completed_preparation_test,reading_score,writing_score,has_passed_math_exam
1,True,group D,5,True,False,93,87,1
4,False,group D,5,True,False,85,86,1
6,False,group E,5,True,False,83,80,1
17,False,group C,1,True,False,92,81,1
18,False,group E,5,False,False,82,85,1


## 6. Generar y visualizar el reporte como texto

`generar_reporte(df)` devuelve un `str` con el número de filas, columnas, duplicados, tipos de datos, valores nulos y estadísticas numéricas. En esta celda el texto se guarda temporalmente en `texto` y se imprime.

In [7]:

# Crear el reporte descriptivo utilizando el DataFrame transformado.
reportes = Reportes()
texto = reportes.generar_reporte(df_transformado)

# Mostrar el reporte generado en la salida del notebook.
print(texto)

REPORTE DEL DATAFRAME
Filas: 261
Columnas: 8
Duplicados: 1

Columnas y tipos:
- is_male: bool
- ethnicity: object
- parental_level_of_education: int64
- has_standard_lunch: bool
- has_completed_preparation_test: bool
- reading_score: int64
- writing_score: int64
- has_passed_math_exam: int64

Valores nulos por columna:
- is_male: 0
- ethnicity: 0
- parental_level_of_education: 0
- has_standard_lunch: 0
- has_completed_preparation_test: 0
- reading_score: 0
- writing_score: 0
- has_passed_math_exam: 0

Estadísticas numéricas:
       parental_level_of_education  reading_score  writing_score  has_passed_math_exam
count                       261.00        261.000        261.000               261.000
mean                          2.59         87.881         86.513                 0.992
std                           1.76          5.652          7.438                 0.087
min                           0.00         80.000         70.000                 0.000
25%                           1.00

## 7. Imprimir el reporte directamente

`imprimir_reporte(df)` ejecuta la generación y muestra el resultado directamente. Es útil para consultar rápidamente el resumen durante una ejecución interactiva.

In [8]:
# Imprimir el resumen descriptivo de los datos transformados.
reportes.imprimir_reporte(df_transformado)

REPORTE DEL DATAFRAME
Filas: 261
Columnas: 8
Duplicados: 1

Columnas y tipos:
- is_male: bool
- ethnicity: object
- parental_level_of_education: int64
- has_standard_lunch: bool
- has_completed_preparation_test: bool
- reading_score: int64
- writing_score: int64
- has_passed_math_exam: int64

Valores nulos por columna:
- is_male: 0
- ethnicity: 0
- parental_level_of_education: 0
- has_standard_lunch: 0
- has_completed_preparation_test: 0
- reading_score: 0
- writing_score: 0
- has_passed_math_exam: 0

Estadísticas numéricas:
       parental_level_of_education  reading_score  writing_score  has_passed_math_exam
count                       261.00        261.000        261.000               261.000
mean                          2.59         87.881         86.513                 0.992
std                           1.76          5.652          7.438                 0.087
min                           0.00         80.000         70.000                 0.000
25%                           1.00

## 8. Guardar el reporte en un archivo de texto

`guardar_reporte(df, ruta)` crea los directorios necesarios y escribe el reporte en UTF-8. El archivo quedará en `src/sincronicas/reportes/fast_food_restaurants.txt` cuando se ejecute desde `src/sincronicas`.

In [9]:
# Guardar el reporte correspondiente al DataFrame transformado.
ruta = reportes.guardar_reporte(
    df_transformado,
    "reportes/student_performance.txt",
)

# Mostrar la ubicación del archivo generado.
print(f"Reporte guardado en: {ruta}")

Reporte guardado en: reportes/student_performance.txt
